# Multimodal Crack and Acoustic Anomaly Detection using ResNet50 and Mel Spectrogram CNN

This notebook implements the multimodal fusion stage of the structural anomaly detection system. The system combines two sensing modalities: visual inspection of concrete surfaces and acoustic analysis of machine sound signals. The visual branch uses a pretrained ResNet50 convolutional neural network to extract high-level image features representing crack patterns on concrete structures. The acoustic branch processes Mel spectrogram representations of audio signals using a CNN model to capture frequency-domain patterns associated with abnormal machine behavior. These two feature representations are concatenated and passed through a fully connected fusion network that learns a joint representation of both modalities. The resulting model produces a final classification indicating structural anomaly presence.

## Environment Setup

This section initializes the development environment by importing the libraries required for implementing the multimodal fusion model and configuring the computational device. PyTorch provides the deep learning framework used to construct and train the neural networks, while torchvision supplies pretrained architectures such as ResNet50 for visual feature extraction. Additional modules such as the PyTorch optimizer and neural network layers are imported to support model construction and training. The code also determines whether a CUDA-enabled GPU is available in the Kaggle runtime environment. If a GPU is detected, computations will be performed on the GPU to accelerate matrix operations and model inference.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
print("GPU count:", torch.cuda.device_count())

torch.backends.cudnn.benchmark = True

## Load Pretrained Image and Audio Models

This section loads the pretrained models that were previously trained for the individual modalities. The visual branch uses a ResNet50 architecture that was trained on the concrete crack dataset. The final classification layer of ResNet50 is replaced with an identity layer so the network outputs the 2048-dimensional visual feature representation instead of a classification prediction. The acoustic branch loads the trained CNN model that processes Mel spectrogram representations of machine sound recordings. Both models are transferred to the configured device and set to evaluation mode so they act as feature extractors for the multimodal fusion network.

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self):
        super(AudioCNN, self).__init__()
        self.conv1 = nn.Conv2d(1,16,3,padding=1)
        self.conv2 = nn.Conv2d(16,32,3,padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.adapt = nn.AdaptiveAvgPool2d((8,8))
        self.fc1 = nn.Linear(32*8*8,128)
        self.fc2 = nn.Linear(128,2)

    def forward(self,x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.adapt(x)
        x = x.view(x.size(0), -1)
        features = torch.relu(self.fc1(x))
        output = self.fc2(features)
        return output

image_model = models.resnet50()
num_features = image_model.fc.in_features
image_model.fc = nn.Identity()
image_model.load_state_dict(
    torch.load("/kaggle/working/resnet50_crack_model.pth")
)
image_model = image_model.to(device)
image_model.eval()

audio_model = AudioCNN()
audio_model.load_state_dict(
    torch.load("/kaggle/working/audio_cnn_slider_model.pth")
)
audio_model = audio_model.to(device)
audio_model.eval()

print("Pretrained models loaded successfully.")

## Multimodal Fusion Architecture

The multimodal fusion model integrates the feature representations extracted from the visual and acoustic branches. The ResNet50 backbone produces a 2048-dimensional visual feature vector, while the audio CNN produces a 128-dimensional acoustic feature vector. These two vectors are concatenated to form a 2176-dimensional multimodal representation. This combined feature vector is then processed by fully connected layers that learn correlations between the two modalities. The fusion network ultimately produces a final classification output indicating whether a structural anomaly is detected.

In [ ]:
class MultimodalModel(nn.Module):
    def __init__(self, image_model, audio_model):
        super(MultimodalModel, self).__init__()
        self.image_model = image_model
        self.audio_model = audio_model
        self.fc1 = nn.Linear(2048 + 128, 256)
        self.fc2 = nn.Linear(256, 2)

    def forward(self, image, audio):
        img_features = self.image_model(image)
        audio_features = self.audio_model(audio)
        combined = torch.cat((img_features, audio_features), dim=1)
        x = torch.relu(self.fc1(combined))
        output = self.fc2(x)
        return output

## Initialization of the Multimodal Fusion Model and Training Configuration

This section initializes the multimodal fusion network and configures the training parameters required for optimizing the model. The fusion architecture combines feature representations extracted from the visual and acoustic branches, producing a joint feature vector that captures information from both modalities. After defining the fusion model, it is transferred to the configured computation device to ensure that training operations are performed efficiently using GPU acceleration when available. In addition, the loss function and optimizer are defined. CrossEntropyLoss is used because the final task is binary classification, while the Adam optimizer is selected for its adaptive learning rate and stable convergence properties during deep neural network training.

In [ ]:
multimodal_model = MultimodalModel(image_model, audio_model)
multimodal_model = multimodal_model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    multimodal_model.parameters(),
    lr=1e-4
)

## Dataset Paths and Multimodal Dataset Class

This section defines the dataset structure used for training the multimodal model. Each training sample consists of a concrete surface image, its corresponding acoustic spectrogram representation, and a class label indicating the presence or absence of a structural anomaly. A custom PyTorch Dataset class is implemented to load both modalities simultaneously. The dataset class reads the image data, loads the spectrogram representation, and returns both inputs together as tensors. This ensures that the fusion model receives synchronized multimodal inputs during training and evaluation. Using a custom dataset class allows efficient batching and integration with PyTorch DataLoader utilities.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import numpy as np

class MultimodalDataset(Dataset):
    def __init__(self, image_paths, spectrogram_paths, labels, transform=None):
        self.image_paths = image_paths
        self.spectrogram_paths = spectrogram_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        spectrogram = np.load(self.spectrogram_paths[idx])
        if self.transform:
            image = self.transform(image)
        spectrogram = torch.tensor(spectrogram).unsqueeze(0).float()
        label = torch.tensor(self.labels[idx]).long()
        return image, spectrogram, label

## Image Transformations

Before feeding images into the ResNet50 backbone, preprocessing operations are applied to ensure that the images match the format expected by the pretrained network. Images are resized to 224×224 pixels, which is the standard input size for ResNet architectures. The pixel values are also normalized using the ImageNet mean and standard deviation values. These normalization parameters align the dataset distribution with the distribution used during the original training of the pretrained network, improving feature extraction performance and ensuring stable inference.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

## Create Dataset and DataLoaders

This section initializes the training and validation datasets and creates DataLoader objects to efficiently feed batches of multimodal data to the fusion model. DataLoaders allow batching, shuffling, and parallel data loading to improve training performance. During training, the dataset is shuffled so that the model does not learn ordering patterns from the data. The validation DataLoader does not shuffle the dataset, ensuring consistent evaluation across epochs. Batching allows the GPU to process multiple multimodal samples simultaneously, which significantly accelerates training.

In [ ]:
train_dataset = MultimodalDataset(
    train_image_paths,
    train_spec_paths,
    train_labels,
    transform
)

val_dataset = MultimodalDataset(
    val_image_paths,
    val_spec_paths,
    val_labels,
    transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

## Fusion Training Loop

This section implements the training loop for the multimodal fusion model. During each epoch, batches of image and spectrogram inputs are passed through the model to generate predictions. The predicted outputs are compared with the ground truth labels using the defined loss function. The computed loss is then propagated backward through the network to update the fusion layer weights using the optimizer. This iterative process enables the model to learn joint representations of visual and acoustic features that improve anomaly detection performance.

In [ ]:
epochs = 10
train_losses = []

for epoch in range(epochs):
    multimodal_model.train()
    running_loss = 0
    for images, specs, labels in tqdm(train_loader):
        images = images.to(device)
        specs = specs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = multimodal_model(images, specs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)

    print(f"Epoch {epoch+1}/{epochs} Loss: {epoch_loss:.4f}")

In [ ]:
torch.save(
    multimodal_model.state_dict(),
    "/kaggle/working/multimodal_fusion_model.pth"
)
print("Fusion model saved successfully.")

## Validation & Evaluation Loop
The validation loop evaluates the performance of the fusion model on unseen data after each training phase. During validation, gradient computation is disabled to reduce memory usage and improve inference speed. The model generates predictions for each validation batch, and these predictions are compared with the true labels to assess the model’s generalization capability. Validation metrics help determine whether the model is overfitting or learning meaningful multimodal representations.

In [ ]:
multimodal_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, specs, labels in val_loader:
        images = images.to(device)
        specs = specs.to(device)
        outputs = multimodal_model(images, specs)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

## Evaluation Metrics and Training Performance Visualization

This section evaluates the performance of the trained multimodal fusion model and visualizes the training dynamics. Quantitative metrics such as accuracy, precision, recall, and F1-score are computed using the predicted labels and ground truth labels obtained from the validation dataset. These metrics provide insight into how effectively the fusion model detects anomalies by combining visual and acoustic information. In addition to numerical evaluation, a confusion matrix is plotted to illustrate the distribution of correct and incorrect predictions across classes. Finally, the training loss curve is visualized across epochs, allowing analysis of the model’s learning behavior and convergence during the training process.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print(classification_report(all_labels, all_preds))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal","Anomaly"],
    yticklabels=["Normal","Anomaly"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Multimodal Confusion Matrix")
plt.show()

plt.figure(figsize=(8,5))
plt.plot(train_losses)
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

## 